# RoadSafe Analytics
## 02 — Data Cleaning and Preprocessing

This notebook cleans the STATS19 collision, vehicle, and casualty datasets.
The original files in `data/raw/` are never modified. Cleaned files are saved in `data/processed/`.

### Cleaning objectives

- Standardise column names
- Remove completely empty rows and columns
- Remove duplicate records
- Standardise missing-value markers
- Remove extra spaces from text values
- Convert numeric-looking columns to numeric types
- Export cleaned datasets for later modelling

In [ ]:
from pathlib import Path
import re
import pandas as pd

# Find the project root from the notebook location
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

raw_dir = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

print('Project root:', project_root)
print('Raw directory exists:', raw_dir.exists())
print('Processed directory:', processed_dir)

In [ ]:
raw_files = sorted(raw_dir.glob('*.csv'))
if not raw_files:
    raise FileNotFoundError(f'No CSV files found in {raw_dir}')

print('CSV files found:')
for file in raw_files:
    print('-', file.name)

In [ ]:
def clean_column_name(name):
    name = str(name).strip().lower()
    name = re.sub(r'[^a-z0-9]+', '_', name)
    return name.strip('_')

def clean_dataframe(df):
    original_rows, original_columns = df.shape

    # Standardise column names
    df.columns = [clean_column_name(column) for column in df.columns]

    # Treat common placeholder values as missing
    missing_values = ['', ' ', 'na', 'n/a', 'null', 'none', '-']
    df = df.replace(missing_values, pd.NA)

    # Remove completely empty rows and columns
    df = df.dropna(axis=0, how='all')
    df = df.dropna(axis=1, how='all')

    # Remove duplicate records
    df = df.drop_duplicates().copy()

    # Trim spaces from text columns
    text_columns = df.select_dtypes(include=['object', 'string']).columns
    for column in text_columns:
        df[column] = df[column].astype('string').str.strip()

    # Convert columns that are entirely numeric to numeric dtype
    for column in df.columns:
        converted = pd.to_numeric(df[column], errors='coerce')
        non_missing = df[column].notna().sum()
        if non_missing > 0 and converted.notna().sum() == non_missing:
            df[column] = converted

    report = {
        'rows_before': original_rows,
        'rows_after': len(df),
        'columns_before': original_columns,
        'columns_after': len(df.columns),
        'duplicate_rows_removed': original_rows - len(df)
    }
    return df, report

In [ ]:
cleaning_report = []

for raw_file in raw_files:
    df = pd.read_csv(raw_file, low_memory=False)
    cleaned_df, report = clean_dataframe(df)

    output_name = raw_file.stem + '_clean.csv'
    output_file = processed_dir / output_name
    cleaned_df.to_csv(output_file, index=False)

    report['file'] = raw_file.name
    report['output'] = output_name
    cleaning_report.append(report)
    print(f'Saved: {output_file}')

report_df = pd.DataFrame(cleaning_report)
report_df

In [ ]:
print('Processed files:')
for file in sorted(processed_dir.glob('*.csv')):
    print(f'- {file.name}: {file.stat().st_size:,} bytes')

print('\nCleaning completed successfully.')

### Validation

The raw files remain unchanged. The cleaned files are available in `data/processed/` and can be used by later project stages.